In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

In [2]:
def load_profile(path):
    """Load a trtexec --exportProfile JSON into a DataFrame.
    Element [0] is a header {"count": N} (iteration count) — skip it.
    Every real entry has: name, timeMs, averageMs, medianMs, percentage."""
    with open(path) as f:
        raw = json.load(f)

    # header is the element WITHOUT a 'name' key ({"count": N})
    count = next((e["count"] for e in raw if "count" in e), None)
    entries = [e for e in raw if "name" in e]        # drops the {"count":…} header

    df = pd.DataFrame(entries)
    # enforce column order / dtypes
    df = df[["name", "timeMs", "averageMs", "medianMs", "percentage"]]
    for col in ["timeMs", "averageMs", "medianMs", "percentage"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    print(f"{Path(path).name}: {len(df)} kernels  (profiled iterations = {count})")
    return df

In [7]:
import os
for root, dirs, files in os.walk("/workspace"):
    for f in files:
        if f.endswith(".json") and "per_kernel" in f.lower():
            print(os.path.join(root, f))

/workspace/Per_kernel_json_file/qat_batch32_per_kernel_profile.json
/workspace/Per_kernel_json_file/ptq_int8_per_kernel_profile.json


In [8]:
QAT_PATH = "/workspace/Per_kernel_json_file/qat_batch32_per_kernel_profile.json"
PTQ_PATH = "/workspace/Per_kernel_json_file/ptq_int8_per_kernel_profile.json"

df_qat = load_profile(QAT_PATH)   # expect 245 kernels
df_ptq = load_profile(PTQ_PATH)   # expect 189 kernels
df_qat.head(10)

qat_batch32_per_kernel_profile.json: 245 kernels  (profiled iterations = 3198)
ptq_int8_per_kernel_profile.json: 189 kernels  (profiled iterations = 3294)


,name,timeMs,averageMs,medianMs,percentage
0,__myl_MulMinMaxRounCast_myl0_0,24.6833,0.007718,0.007680,0.410286
1,model.0.conv.weight + /model.0/conv/weight_quantizer/QuantizeLinear + /model...,53.4611,0.016717,0.016640,0.888631
2,model.1.conv.weight + /model.1/conv/weight_quantizer/QuantizeLinear + /model...,40.2967,0.012601,0.012576,0.669812
3,model.2.cv1.conv.weight + /model.2/cv1/conv/weight_quantizer/QuantizeLinear ...,37.5717,0.011749,0.011680,0.624517
4,"PWN(/model.2/cv1/act/Sigmoid, /model.2/cv1/act/Mul)",24.3234,0.007606,0.007584,0.404304
5,__myl_MulMinMaxRounCastReplConcReshTran_myl5_0,25.7250,0.008044,0.008032,0.427601
6,model.2.m.0.cv1.conv.weight + /model.2/m.0/cv1/conv/weight_quantizer/Quantiz...,38.2750,0.011968,0.011904,0.636208
7,model.2.m.0.cv2.conv.weight + /model.2/m.0/cv2/conv/weight_quantizer/Quantiz...,41.7801,0.013064,0.013056,0.694470
8,__myl_MulMinMaxRounCast_myl8_0,21.4347,0.006703,0.006656,0.356288
9,"PWN(PWN(/model.2/m.0/cv2/act/Sigmoid, /model.2/m.0/cv2/act/Mul), /model.2/m....",24.2405,0.007580,0.007552,0.402926


In [9]:
def common_columns(df_a, df_b):
    common = sorted(set(df_a.columns) & set(df_b.columns))
    only_a = sorted(set(df_a.columns) - set(df_b.columns))
    only_b = sorted(set(df_b.columns) - set(df_a.columns))
    print(f"Common columns ({len(common)}): {common}")
    if only_a: print(f"Only in QAT: {only_a}")
    if only_b: print(f"Only in PTQ: {only_b}")
    return common

common_cols = common_columns(df_qat, df_ptq)   # should be all 5: name, timeMs, averageMs, medianMs, percentage

Common columns (5): ['averageMs', 'medianMs', 'name', 'percentage', 'timeMs']


In [10]:
df_qat_c = df_qat[common_cols].copy()
df_ptq_c = df_ptq[common_cols].copy()
df_qat_c["engine"] = "QAT"
df_ptq_c["engine"] = "PTQ"

print("QAT:", df_qat_c.shape, "| PTQ:", df_ptq_c.shape)

QAT: (245, 6) | PTQ: (189, 6)


In [11]:
def summarize(df, label):
    print(f"\n=== {label}  ({len(df)} kernels) ===")
    print(f"  sum(averageMs)   = {df['averageMs'].sum():.4f} ms   <- per-inference total")
    print(f"  sum(medianMs)    = {df['medianMs'].sum():.4f} ms")
    print(f"  sum(percentage)  = {df['percentage'].sum():.2f} %")
    print(f"  top-5 kernels by averageMs:")
    top = df.nlargest(5, "averageMs")[["name", "averageMs", "percentage"]]
    for _, r in top.iterrows():
        print(f"    {r['averageMs']:.4f} ms ({r['percentage']:.1f}%)  {r['name'][:70]}")

summarize(df_qat_c, "QAT")
summarize(df_ptq_c, "PTQ")


=== QAT  (245 kernels) ===
  sum(averageMs)   = 1.8812 ms   <- per-inference total
  sum(medianMs)    = 1.8751 ms
  sum(percentage)  = 100.00 %
  top-5 kernels by averageMs:
    0.0348 ms (1.8%)  __myl_Topk_myl229_2
    0.0167 ms (0.9%)  model.0.conv.weight + /model.0/conv/weight_quantizer/QuantizeLinear + 
    0.0131 ms (0.7%)  model.2.m.0.cv2.conv.weight + /model.2/m.0/cv2/conv/weight_quantizer/Q
    0.0130 ms (0.7%)  model.23.one2one_cv2.2.0.conv.weight + /model.23/one2one_cv2.2/one2one
    0.0126 ms (0.7%)  model.1.conv.weight + /model.1/conv/weight_quantizer/QuantizeLinear + 

=== PTQ  (189 kernels) ===
  sum(averageMs)   = 1.6081 ms   <- per-inference total
  sum(medianMs)    = 1.6018 ms
  sum(percentage)  = 100.00 %
  top-5 kernels by averageMs:
    0.0582 ms (3.6%)  __myl_Topk_myl183_2
    0.0188 ms (1.2%)  /model.10/m/m.0/attn/MatMul_1
    0.0188 ms (1.2%)  /model.22/m.0/m.0.1/attn/MatMul_1
    0.0167 ms (1.0%)  /model.0/conv/Conv + PWN(PWN(/model.0/act/Sigmoid), PWN(/model.0

In [12]:
def op_category(name):
    """Bucket each kernel by operation type, from its name."""
    n = name.lower()
    if "pwn(sigmoid" in n or "silu" in n:
        if "conv" in n:
            return "conv+SiLU (fused)"
        return "SiLU standalone"
    if "conv" in n:
        return "conv only"
    if "reformat" in n or "shuffle" in n:
        return "reformat"
    if "topk" in n or "nms" in n or "gather" in n:
        return "NMS/TopK"
    if "softmax" in n or "attn" in n or "matmul" in n:
        return "attention"
    return "other"

for df in (df_qat_c, df_ptq_c):
    df["op_category"] = df["name"].apply(op_category)

# op-category breakdown per engine (this is the conv+SiLU fusion story)
op_qat = df_qat_c.groupby("op_category").agg(ms=("averageMs","sum"), n=("averageMs","size"))
op_ptq = df_ptq_c.groupby("op_category").agg(ms=("averageMs","sum"), n=("averageMs","size"))
op_compare = op_qat.join(op_ptq, lsuffix="_QAT", rsuffix="_PTQ", how="outer").fillna(0)
op_compare["Δms"] = op_compare["ms_QAT"] - op_compare["ms_PTQ"]
op_compare.round(4)

,ms_QAT,n_QAT,ms_PTQ,n_PTQ,Δms
op_category,,,,,
NMS/TopK,0.0447,2,0.0713,2,-0.0265
attention,0.0432,8,0.1154,18,-0.0722
conv only,0.9890,109,1.0156,109,-0.0266
other,0.7655,120,0.3940,58,0.3715
reformat,0.0388,6,0.0118,2,0.0269
